# i need to change my design like cpu -> gpu structure, not gpu dedicated design

In [ ]:
import torch
import torch.nn as nn
import optuna
import pandas as pd
from optuna import Trial
import torchmetrics

torch.manual_seed(42)
device = "cuda"

## 13

In [ ]:
x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)

f = torch.sin((x**2)*y)
f.backward()

print("Gradient of x:", x.grad)
print("Gradient of y:", y.grad)


## 14

In [ ]:
class InhuDense(nn.Module):
    def __init__(self, input_num:int, output_num:int):
        super().__init__()
        self.input_w = nn.Parameter(torch.randn(input_num, output_num, device=device))
        self.input_b = nn.Parameter(torch.randn(output_num, device=device))
    
    def forward(self, x):
        return torch.relu(x @ self.input_w + self.input_b)

## 15 classification

In [ ]:
from sklearn.datasets import fetch_covtype
from torch.utils.data import Dataset

# Load the Covertype dataset
covtype = fetch_covtype()
X = covtype['data']
y = covtype['target']

# Create a custom PyTorch Dataset
# NOTE: tensors stay on CPU here. The DataLoader serves CPU batches and the
# training loop moves each batch to the GPU (cpu -> dataloader -> gpu).
class CovtypeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)   # CPU
        self.y = torch.tensor(y - 1, dtype=torch.long)  # CPU
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

covtype_dataset = CovtypeDataset(X, y)

covtype_df = pd.DataFrame(X, columns=covtype['feature_names'])
covtype_df

In [ ]:
np.unique_counts(y)

In [ ]:
batch_size = 128
# In-RAM tensor dataset has no disk I/O to parallelize, and on Windows
# notebooks worker processes (spawn) must re-pickle the whole dataset, so
# num_workers=0 is the right choice here. persistent_workers requires
# num_workers > 0, so it stays False.
num_worker = 0
persistent_worker = False
# pin_memory=True lets the DataLoader place each CPU batch in page-locked
# memory, which makes the .to(device, non_blocking=True) transfer async.
pin_memory = True

n_epoch = 5
lr = 0.001

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

data_train, data_val, data_test  = random_split(covtype_dataset, [0.7, 0.2, 0.1])

train_loader = DataLoader(data_train, shuffle=True, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)
val_loader = DataLoader(data_val, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)
test_loader = DataLoader(data_test, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)

In [ ]:
from torch.optim import Optimizer

class Classification(nn.Module):
    def __init__(self, n_inputs:int = 54, n_hidden_num_list:list[int] = [], n_classes = 7) -> None:
        super().__init__()
        hidden_layers:list[nn.Module] = []
        for idx in range(1,len(n_hidden_num_list)):
            new_layer = nn.Linear(n_hidden_num_list[idx-1], n_hidden_num_list[idx])
            hidden_layers.append(new_layer)
            hidden_layers.append(nn.ReLU())

        self.mlp = nn.Sequential(
            nn.Linear(n_inputs, n_hidden_num_list[0]), nn.ReLU(),
            *hidden_layers,
            nn.Linear(n_hidden_num_list[-1], n_classes)
        )
    
    def forward(self, X):
        return self.mlp(X)


def train(model:nn.Module, criterion, optimizer:Optimizer, dataloader:DataLoader, n_epoch:int = 1):
    model.to(device)
    model.train()
    for i in range(n_epoch):
        total_loss:float = 0.0
        for X,y in dataloader:
            optimizer.zero_grad()
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            total_loss += loss.item()
            optimizer.step()
            
        print(f"epoch:{i}, mean_loss:{total_loss/len(dataloader)}")
    
def evaluate(model:nn.Module, metric_fn, dataloader:DataLoader, aggre_fn = torch.mean):
    model.to(device)
    model.eval()
    metrics = []
    for X,y in dataloader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.no_grad():
            y_pred = model(X)
            metrics.append(metric_fn(y_pred, y))
    
    metric_fn.reset()
    return aggre_fn(torch.stack(metrics))

In [ ]:
def objective(trial:Trial, criterion, dataloader:DataLoader):
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    layer_nums = trial.suggest_int("layer_nums", 1,4)
    n_hidden_num_list = [trial.suggest_int(f"n_hidden_{i}", 32, 512) for i in range(layer_nums)]
    
    model = Classification(n_hidden_num_list=n_hidden_num_list).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

    for i in range(n_epoch):
        train(model, criterion, optimizer, dataloader, n_epoch=1)
        val_acc = evaluate(model, accuracy, val_loader)
        trial.report(val_acc.item(), step=i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return val_acc.item()

In [ ]:
from backend.engine import engine, db_url

criterion = nn.CrossEntropyLoss()
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, interval_steps=1)

storage = optuna.storages.RDBStorage(url = db_url)
study = optuna.create_study(
    study_name = "covtype_study",
    storage=storage,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    load_if_exists=False
)

In [ ]:
study.optimize(lambda trial: objective(trial, criterion, train_loader), n_trials=20, n_jobs=1)

print("Best trial:")
print(f"  Accuracy: {study.best_value:.4f}")
print(f"  Params:   {study.best_params}")

In [ ]:
best_params = study.best_params
best_lr = best_params["lr"]
best_hidden = [best_params[f"n_hidden_{i}"] for i in range(best_params["layer_nums"])]

best_model = Classification(n_hidden_num_list=best_hidden)
optimizer = torch.optim.AdamW(best_model.parameters(), lr=best_lr)
train(best_model, criterion, optimizer, train_loader, n_epoch)

torch.save({
    "model_state_dict": best_model.state_dict(),
    "hidden_layers": best_hidden,
    "lr": best_lr,
}, "best_model.pth")
print("Model saved to best_model.pth")

In [ ]:
metric_fn = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
evaluate(best_model, metric_fn, test_loader)